# scDesign3 null simulations: no spatial gradient (DLPFC 151673, Layer3)

Fit an intercept-only scDesign3 model to the 151673 `Layer3` spots and simulate datasets with **no spatial dependence** of the mean:

- marginals: negative binomial, `mu_formula = "1"`, `sigma_formula = "1"` (one mean and one dispersion per gene),
- dependence: a single Gaussian copula over all genes (`corr_formula = "1"`), so gene-gene correlation is kept,
- simulated spots are placed on a `sqrt(N) x sqrt(N)` grid. Since the model ignores location, any placement is a valid null.

Python handles I/O and plotting; R (scDesign3) runs in `%%R` cells through rpy2, and `anndata2ri` converts AnnData to `SingleCellExperiment`.

scDesign3 requires at least one of `celltype` / `pseudotime` / `spatial`, so a constant one-level `domain` column is passed as `celltype`. It does not enter the model because both formulas are `"1"`.

Smoke-test mode: set `SCDESIGN3_SMOKE=1` (and optionally `SCDESIGN3_OUT_DIR`) in the environment before running.

In [ ]:
import os
import sys
from pathlib import Path

REPO = Path("/scratch/jhu/uchitra1/users/ajain71/isodepth-statistical-testing")
R_ENV = Path("/home/jhu/ajain71/miniforge3/envs/r_env")
os.environ["R_HOME"] = str(R_ENV / "lib" / "R")
os.environ["PATH"] = f"{R_ENV / 'bin'}:{os.environ['PATH']}"
sys.path.insert(0, str(REPO))

%load_ext rpy2.ipython

R packages must be loaded before any Python import that pulls in torch (scanpy, the project loader): torch bundles an older `libgomp` that breaks R's `SparseArray.so` if it is loaded first.

In [ ]:
%%R
suppressPackageStartupMessages({
    library(scDesign3)
    library(SingleCellExperiment)
})
cat("scDesign3", as.character(packageVersion("scDesign3")), "\n")

In [ ]:
import anndata as ad
import anndata2ri
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp

from data.h5ad_loader import _select_hvg_mask

a2r = anndata2ri.converter

## Parameters

In [ ]:
REF_H5AD = REPO / "data/h5ad/151673.h5ad"
DOMAIN_KEY, DOMAIN = "Region", "Layer3"

G = 200                  # genes
N = 1024                 # simulated spots per replicate (must be a perfect square)
N_REPS = 10              # independent null replicates from one fitted model
MIN_FRAC_DETECTED = 0.10 # gene must be nonzero in >= this fraction of reference spots
SEED = 42
N_CORES = 4
OUT_DIR = REPO / "data/h5ad/scdesign3_null"

if os.environ.get("SCDESIGN3_SMOKE") == "1":
    G, N, N_REPS, N_CORES = 20, 100, 1, 1
    OUT_DIR = Path("/tmp/scdesign3_null_smoke")
OUT_DIR = Path(os.environ.get("SCDESIGN3_OUT_DIR", OUT_DIR))
OUT_DIR.mkdir(parents=True, exist_ok=True)

SIDE = int(round(np.sqrt(N)))
assert SIDE * SIDE == N, "N must be a perfect square for the grid layout"
PREFIX = f"dlpfc151673_{DOMAIN.lower()}_null_G{G}_N{N}"
MODEL_RDS = str(OUT_DIR / f"{PREFIX}_model.rds")
print(PREFIX, "->", OUT_DIR)

## Reference: Layer3 spots, G highly variable genes

Genes are first filtered by detection rate (stable NB fits), then the top-`G` Seurat HVGs are taken using the project's loader helper (computed on a normalize+log1p copy of the counts).

In [ ]:
adata = ad.read_h5ad(REF_H5AD)
adata.var_names_make_unique()
adata = adata[adata.obs[DOMAIN_KEY] == DOMAIN].copy()

X = adata.X.toarray() if sp.issparse(adata.X) else np.asarray(adata.X)
assert np.allclose(X, np.round(X)), "reference X must hold raw integer counts"
adata.X = sp.csr_matrix(X)

detected = (X > 0).mean(axis=0) >= MIN_FRAC_DETECTED
adata = adata[:, detected].copy()
hvg = _select_hvg_mask(adata, G)
adata = adata[:, hvg].copy()
assert adata.n_vars == G, f"HVG selection returned {adata.n_vars} genes, expected {G}"

ref_counts = np.rint(adata.X.toarray()).astype(np.int32)  # spots x genes
genes = adata.var_names.to_list()
adata_ref = ad.AnnData(
    X=ref_counts,
    obs=pd.DataFrame({"domain": pd.Categorical([DOMAIN] * adata.n_obs)}, index=adata.obs_names),
    var=pd.DataFrame(index=genes),
)
adata_ref.layers["counts"] = ref_counts.copy()

print(f"{adata_ref.n_obs} spots x {adata_ref.n_vars} genes "
      f"({int(detected.sum())} genes passed detection >= {MIN_FRAC_DETECTED:.0%})")
print(f"zero fraction: {(ref_counts == 0).mean():.3f}, "
      f"median per-gene mean: {np.median(ref_counts.mean(0)):.2f}")

## Fit scDesign3 (R)

The fit is done once with scDesign3's step functions (`construct_data`, `fit_marginal`, `fit_copula`), equivalent to `scdesign3(..., mu_formula = "1", sigma_formula = "1", family_use = "nb", corr_formula = "1", copula = "gaussian", return_model = TRUE)`, so every replicate is drawn from the same fitted model.

In [ ]:
%%R -i adata_ref -i N_CORES -i MODEL_RDS -c a2r
sce_ref <- adata_ref
n_cores <- as.integer(N_CORES)

dat <- construct_data(
    sce = sce_ref, assay_use = "counts",
    celltype = "domain", pseudotime = NULL, spatial = NULL, other_covariates = NULL,
    corr_by = "1"
)
marginals <- fit_marginal(
    data = dat, predictor = "gene",
    mu_formula = "1", sigma_formula = "1",
    family_use = "nb", usebam = FALSE,
    n_cores = n_cores, parallelization = "mcmapply"
)
copula <- fit_copula(
    sce = sce_ref, assay_use = "counts", input_data = dat$dat,
    marginal_list = marginals, family_use = "nb",
    copula = "gaussian", important_feature = "all",
    n_cores = n_cores, parallelization = "mcmapply"
)
saveRDS(list(dat = dat, marginals = marginals, copula = copula), MODEL_RDS)
cat("fitted", length(marginals), "marginals; copula groups:", names(copula$copula_list), "\n")

## Simulate N spots x N_REPS replicates (R)

`new_covariate` has N rows with the single `domain` level and a single copula group. Under the intercept-only model `extract_para` returns the same per-gene parameters for every spot. Replicate `k` uses `set.seed(SEED + k)`.

In [ ]:
%%R -i N -i N_REPS -i SEED -o sim_array -o sim_genes -o fit_mean -o fit_sigma
N <- as.integer(N); N_REPS <- as.integer(N_REPS); SEED <- as.integer(SEED)
new_cov <- data.frame(
    domain = factor(rep(levels(colData(sce_ref)$domain), length.out = N)),
    corr_group = factor(rep(1, N))
)
para <- extract_para(
    sce = sce_ref, assay_use = "counts", marginal_list = marginals,
    family_use = "nb", new_covariate = new_cov, data = dat$dat,
    n_cores = n_cores, parallelization = "mcmapply"
)
sims <- lapply(seq_len(N_REPS), function(k) {
    set.seed(SEED + k)
    simu_new(
        sce = sce_ref, assay_use = "counts",
        mean_mat = para$mean_mat, sigma_mat = para$sigma_mat, zero_mat = para$zero_mat,
        quantile_mat = NULL, copula_list = copula$copula_list,
        family_use = "nb", nonnegative = TRUE, nonzerovar = FALSE,
        input_data = dat$dat, new_covariate = new_cov,
        important_feature = copula$important_feature, filtered_gene = dat$filtered_gene,
        n_cores = n_cores, parallelization = "mcmapply"
    )
})
sim_genes <- rownames(sims[[1]])
sim_array <- array(unlist(lapply(sims, as.matrix)), dim = c(length(sim_genes), N, N_REPS))  # genes x spots x reps
fit_mean <- as.numeric(para$mean_mat[1, sim_genes])
fit_sigma <- as.numeric(para$sigma_mat[1, sim_genes])
cat("simulated", dim(sim_array), "\n")

## Write h5ad replicates

Counts go in `X` (spots x genes, float32, integer-valued like the 151673 reference). The grid goes in `obsm['spatial']`, with spacing 1 and arbitrary units, so use `coordinate_um_per_unit = 1.0`.

In [ ]:
sim_array = np.asarray(sim_array)
sim_genes = [str(g) for g in sim_genes]
assert sim_genes == genes, "simulated gene order differs from reference"
assert sim_array.shape == (G, N, N_REPS)
assert np.allclose(sim_array, np.round(sim_array)) and sim_array.min() >= 0

row, col = np.divmod(np.arange(N), SIDE)
grid = np.column_stack([col, row]).astype(np.float32)
provenance = {
    "reference_h5ad": str(REF_H5AD), "domain_key": DOMAIN_KEY, "domain": DOMAIN,
    "n_reference_spots": int(adata_ref.n_obs), "min_frac_detected": MIN_FRAC_DETECTED,
    "gene_selection": "seurat_hvg_on_normalize_log1p",
    "mu_formula": "1", "sigma_formula": "1", "corr_formula": "1",
    "family_use": "nb", "copula": "gaussian", "model_rds": MODEL_RDS,
    "layout": f"grid_{SIDE}x{SIDE}_spacing1",
}

sim_paths = []
for k in range(N_REPS):
    a = ad.AnnData(
        X=sp.csr_matrix(sim_array[:, :, k].T.astype(np.float32)),
        obs=pd.DataFrame({DOMAIN_KEY: pd.Categorical([DOMAIN] * N)},
                         index=[f"spot{i:04d}" for i in range(N)]),
        var=pd.DataFrame(index=genes),
    )
    a.obsm["spatial"] = grid
    a.uns["scdesign3"] = {**provenance, "replicate": k + 1, "seed": SEED + k + 1}
    path = OUT_DIR / f"{PREFIX}_rep{k + 1:02d}.h5ad"
    a.write_h5ad(path)
    sim_paths.append(path)
print("\n".join(str(p) for p in sim_paths))

## Sanity checks: replicate 1 vs reference

The simulated per-gene mean, variance, zero fraction and gene-gene correlations should track the reference. The grid heatmaps should show no spatial structure.

In [ ]:
sim1 = sim_array[:, :, 0].T  # spots x genes
ref = ref_counts.astype(np.float64)

def offdiag_corr(x):
    c = np.corrcoef(np.log1p(x), rowvar=False)
    return c[np.triu_indices_from(c, k=1)]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, f, name in [(axes[0], np.mean, "mean"), (axes[1], np.var, "variance")]:
    r, s = f(ref, axis=0), f(sim1, axis=0)
    ax.loglog(r, s, ".", alpha=0.6)
    lim = [min(r.min(), s.min()) * 0.8, max(r.max(), s.max()) * 1.2]
    ax.plot(lim, lim, "k--", lw=1)
    ax.set(xlabel=f"reference {name}", ylabel=f"simulated {name}", title=f"per-gene {name}")
r0, s0 = (ref == 0).mean(0), (sim1 == 0).mean(0)
axes[2].plot(r0, s0, ".", alpha=0.6)
axes[2].plot([0, 1], [0, 1], "k--", lw=1)
axes[2].set(xlabel="reference zero fraction", ylabel="simulated zero fraction", title="per-gene zero fraction")
bins = np.linspace(-0.4, 0.8, 61)
axes[3].hist(offdiag_corr(ref), bins=bins, alpha=0.5, density=True, label="reference")
axes[3].hist(offdiag_corr(sim1), bins=bins, alpha=0.5, density=True, label="simulated")
axes[3].set(xlabel="gene-gene Pearson r (log1p counts)", title="off-diagonal correlations")
axes[3].legend()
plt.tight_layout()
plt.show()

print(f"fitted vs reference mean, median ratio: {np.median(fit_mean / ref.mean(0)):.3f}")

In [ ]:
show = np.argsort(-sim1.mean(0))[:4]
fig, axes = plt.subplots(1, len(show), figsize=(4 * len(show), 3.6))
for ax, j in zip(axes, show):
    img = np.zeros((SIDE, SIDE))
    img[row, col] = np.log1p(sim1[:, j])
    im = ax.imshow(img, origin="lower", cmap="viridis")
    ax.set_title(f"{genes[j]} (log1p, rep 1)")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## Session info

In [ ]:
%%R
sessionInfo()

In [ ]:
from importlib.metadata import version
print({"python": sys.version.split()[0],
       **{p: version(p) for p in ["anndata", "anndata2ri", "rpy2", "numpy", "scanpy"]}})